# 6.21 - CertCF Adult Simplex Merge Compression Diagnostics

This notebook tests a stricter high-dimensional convex-polytope merge on **Adult**.

Instead of wrapping local anchor groups with an axis-aligned box, we build a **simplex / convex hull of the selected region centers** and run a fresh one-vs-all certification pass on the simplex.

This is intentionally a compression-oriented experiment:
- build one standard Adult CertCF atlas
- propose local groups of 2/3/4 centers
- certify the convex hull of those centers
- keep only certified merges
- report atlas compression and simplex certification statistics

Important note:
- a simplex of centers is much tighter than a box
- but it is also a thinner set, so this notebook is mainly about **certifiability and compression**, not direct coverage volume


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm
from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy
from certcf.certification.wrapping import WrappedModel
from dataset_specs import get_tabular_dataset_spec
from models.classifiers import TabularClassifier
from training.datamodules.adult import AdultDataModule
from training.lit_classifier import LitClassifier

torch.set_grad_enabled(False)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'device': str(DEVICE), 'seed': SEED})


{'device': 'cuda', 'seed': 42}


In [2]:
DATA_CFG = {
    'filepath': str(ROOT / 'data' / 'Adult' / 'raw.parquet'),
    'batch_size': 256,
    'val_fraction': 0.1,
    'test_fraction': 0.1,
    'seed': SEED,
    'num_workers': 0,
    'pca_enabled': False,
}

dm = AdultDataModule(**DATA_CFG)
dm.setup()
X_TRAIN, Y_TRAIN_TRUE = [t.numpy() for t in dm.train_ds.tensors]
X_TEST, Y_TEST_TRUE = [t.numpy() for t in dm.test_ds.tensors]

print({
    'train_shape': tuple(X_TRAIN.shape),
    'test_shape': tuple(X_TEST.shape),
    'train_class_counts': {int(c): int((Y_TRAIN_TRUE == c).sum()) for c in np.unique(Y_TRAIN_TRUE)},
    'test_class_counts': {int(c): int((Y_TEST_TRUE == c).sum()) for c in np.unique(Y_TEST_TRUE)},
})


{'train_shape': (36178, 104), 'test_shape': (4522, 104), 'train_class_counts': {0: 27224, 1: 8954}, 'test_class_counts': {0: 3385, 1: 1137}}


In [3]:
CKPT_PATH = ROOT / 'checkpoints' / 'adult_classifier' / 'best.ckpt'
SPEC = get_tabular_dataset_spec('adult')


def infer_tabular_classifier_dims_from_checkpoint(checkpoint: str | Path) -> tuple[list[int], int]:
    ckpt = torch.load(str(checkpoint), map_location='cpu', weights_only=False)
    state_dict = ckpt.get('state_dict', {})
    hidden_1 = state_dict.get('model.net.4.weight')
    output = state_dict.get('model.net.6.weight')
    if hidden_1 is None or output is None:
        raise KeyError('Could not infer hidden_dims / num_classes from the Adult checkpoint.')
    hidden_dims = [int(hidden_1.shape[1]), int(hidden_1.shape[0])]
    num_classes = int(output.shape[0])
    return hidden_dims, num_classes


def load_adult_model(checkpoint: str | Path, device: torch.device = DEVICE) -> torch.nn.Module:
    hidden_dims, num_classes = infer_tabular_classifier_dims_from_checkpoint(checkpoint)
    backbone = TabularClassifier(
        input_types=list(SPEC.input_types),
        cardinalities=list(SPEC.cardinalities),
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout=0.2,
    )
    lit = LitClassifier.load_from_checkpoint(str(checkpoint), model=backbone, map_location=str(device))
    net = lit.model.eval().to(device)
    net_no_dropout = torch.nn.Sequential(*[m for m in net.net if not isinstance(m, torch.nn.Dropout)])
    return net_no_dropout.eval().to(device)


@torch.no_grad()
def predict_np(model: torch.nn.Module, x: np.ndarray, device: torch.device = DEVICE) -> tuple[np.ndarray, np.ndarray]:
    x_t = torch.from_numpy(np.asarray(x, dtype=np.float32)).to(device)
    logits = model(x_t)
    probs = torch.softmax(logits, dim=1).cpu().numpy().astype(np.float32)
    preds = probs.argmax(axis=1)
    return preds, probs


MODEL = load_adult_model(CKPT_PATH, device=DEVICE)
Y_TRAIN_PRED, _ = predict_np(MODEL, X_TRAIN)
Y_TEST_PRED, _ = predict_np(MODEL, X_TEST)

print({
    'checkpoint': str(CKPT_PATH),
    'predicted_train_class_counts': {int(c): int((Y_TRAIN_PRED == c).sum()) for c in np.unique(Y_TRAIN_PRED)},
    'predicted_test_class_counts': {int(c): int((Y_TEST_PRED == c).sum()) for c in np.unique(Y_TEST_PRED)},
})


{'checkpoint': '/home/gabrielepintus/Documents/github/PreimageCounterfactualSampling/checkpoints/adult_classifier/best.ckpt', 'predicted_train_class_counts': {0: 28458, 1: 7720}, 'predicted_test_class_counts': {0: 3576, 1: 946}}


## Atlas and simplex-merge setup

We keep a diagnostic support subset and one Adult atlas. Then we try to replace small local groups of anchor regions with a single certified **simplex of centers**.


In [4]:
RUN_CFG = {
    'alpha': 0.45,
    'support_max_per_class': 600,
    'batch_size': 256,
    'norm': 1,
    'distance_norm': 1,
    'solver_maxiter': 500,
    'query_method': 'nearest_anchor',
    'proposal_top_center_neighbors': 20,
    'proposal_neighbor_pool': 12,
    'proposal_max_triples_per_anchor': 10,
    'proposal_max_quads_per_anchor': 8,
    'iter_max_passes': 12,
    'iter_top_pairs_per_pass': 800,
    'iter_top_triples_per_pass': 400,
    'iter_top_quads_per_pass': 250,
}


def stratified_subsample(x: np.ndarray, y: np.ndarray, max_per_class: int | None, seed: int = SEED):
    if max_per_class is None:
        keep = np.arange(len(x))
        return x, y, keep
    rng = np.random.default_rng(seed)
    keep = []
    for cls in sorted(np.unique(y)):
        idx = np.where(y == cls)[0]
        if len(idx) > max_per_class:
            idx = rng.choice(idx, size=max_per_class, replace=False)
        keep.append(np.sort(idx))
    keep = np.sort(np.concatenate(keep))
    return x[keep], y[keep], keep


X_SUPPORT, Y_SUPPORT, SUPPORT_IDX = stratified_subsample(
    X_TRAIN,
    Y_TRAIN_PRED,
    max_per_class=RUN_CFG['support_max_per_class'],
)

print({
    'support_shape': tuple(X_SUPPORT.shape),
    'support_class_counts': {int(c): int((Y_SUPPORT == c).sum()) for c in np.unique(Y_SUPPORT)},
})

DS = TensorDataset(torch.from_numpy(X_SUPPORT).float(), torch.from_numpy(Y_SUPPORT).long())
ATLAS = CertCFAtlas(
    MODEL,
    DS,
    DEVICE,
    cnn=False,
    norm=RUN_CFG['norm'],
    distance_norm=RUN_CFG['distance_norm'],
    lirpa_method='backward',
    eps_strategy=NearestOppositeClassClearanceStrategy(alpha=RUN_CFG['alpha']),
    batch_size=RUN_CFG['batch_size'],
    default_query_method=RUN_CFG['query_method'],
    solver_maxiter=RUN_CFG['solver_maxiter'],
)

t0 = time.perf_counter()
ATLAS.build(build_unions=False, verbose=True)
ATLAS_BUILD_SECONDS = time.perf_counter() - t0
print({'atlas_build_seconds': float(ATLAS_BUILD_SECONDS)})


{'support_shape': (1200, 104), 'support_class_counts': {0: 600, 1: 600}}
Building certified atlas (eps in [0.3619, 10.77] (NearestOppositeClassClearanceStrategy), L1 norm)...
  Computing LiRPA bounds...


Computing bounds:   0%|          | 0/2 [00:00<?, ?it/s]/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/perturbations.py:199: RuntimeWarning: divide by zero encountered in scalar divide
  self.dual_norm = 1 if (norm == np.inf) else (np.float64(1.0) / (1 - 1.0 / self.norm))
/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/operators/linear.py:650: RuntimeWarning: divide by zero encountered in scalar divide
  dual_norm = np.float64(1.0) / (1 - 1.0 / norm)
Computing bounds: 100%|██████████| 2/2 [00:14<00:00,  7.36s/it]

  Building BVH spatial indices...
    Class 0: 600 polytopes, tree depth 11
    Class 1: 600 polytopes, tree depth 11
Done! Total: 1200 polytopes across 2 classes
{'atlas_build_seconds': 14.761442861003161}


In [5]:
def run_lirpa_box_lower_bound(model, target_label, x_L, x_U, device=DEVICE, lirpa_method='backward'):
    x_L = np.asarray(x_L, dtype=np.float32).reshape(1, -1)
    x_U = np.asarray(x_U, dtype=np.float32).reshape(1, -1)
    x_center = 0.5 * (x_L + x_U)

    wrapped = WrappedModel(model, label=int(target_label), device=device, n_labels=2).to(device).to(torch.float32)
    wrapped.eval()

    x_center_t = torch.from_numpy(x_center).to(device)
    x_L_t = torch.from_numpy(x_L).to(device)
    x_U_t = torch.from_numpy(x_U).to(device)
    ptb = PerturbationLpNorm(norm=np.inf, x_L=x_L_t, x_U=x_U_t)
    X_bounded = BoundedTensor(x_center_t, ptb)

    bounded_model = BoundedModule(wrapped, X_bounded)
    _ = bounded_model(X_bounded)

    needed_A = defaultdict(set)
    needed_A[bounded_model.output_name[0]].add(bounded_model.input_name[0])
    _, _, A_dict = bounded_model.compute_bounds(
        x=(X_bounded,),
        method=lirpa_method,
        return_A=True,
        needed_A_dict=needed_A,
    )

    A = A_dict[bounded_model.output_name[0]][bounded_model.input_name[0]]
    lA = A['lA'].detach().cpu().numpy().reshape(-1, x_center.shape[1])
    lbias = A['lbias'].detach().cpu().numpy().reshape(-1)
    return lA[0].astype(np.float64), float(lbias[0])


def certify_simplex_region(model, target_label: int, vertices: np.ndarray):
    vertices = np.asarray(vertices, dtype=np.float32)
    x_L = vertices.min(axis=0)
    x_U = vertices.max(axis=0)
    c, d = run_lirpa_box_lower_bound(model, target_label, x_L=x_L, x_U=x_U)
    values = vertices @ c + d
    min_value = float(values.min())
    widths = x_U - x_L
    return {
        'certified': bool(min_value > 0.0),
        'min_affine_margin': float(min_value),
        'bbox_width_l1': float(widths.sum()),
        'bbox_width_linf': float(widths.max()),
        'x_L': x_L,
        'x_U': x_U,
    }


def make_initial_center_regions(atlas, target_label: int):
    bd = atlas.bounds[target_label]
    regions = []
    for idx in range(len(bd['X'])):
        center = np.asarray(bd['X'][idx], dtype=np.float32)
        regions.append({
            'region_key': f'class{target_label}_poly_{idx}',
            'center': center,
            'vertices': center.reshape(1, -1),
            'member_count': 1,
            'source_ids': [int(idx)],
        })
    return regions


def build_local_simplex_candidates(regions, cfg):
    if len(regions) < 2:
        return pd.DataFrame(), {}

    centers = np.stack([np.asarray(region['center'], dtype=np.float32) for region in regions], axis=0)
    center_dmat = np.abs(centers[:, None, :] - centers[None, :, :]).sum(axis=2)
    candidate_map = {}

    def register_subset(indices, source_tag):
        indices = tuple(sorted({int(idx) for idx in indices}))
        if len(indices) < 2:
            return
        key = '|'.join(regions[idx]['region_key'] for idx in indices)
        if key in candidate_map:
            candidate_map[key]['proposal_sources'].add(source_tag)
            return
        subset = [regions[idx] for idx in indices]
        vertices = np.stack([region['center'] for region in subset], axis=0)
        x_L = vertices.min(axis=0)
        x_U = vertices.max(axis=0)
        widths = x_U - x_L
        candidate_map[key] = {
            'candidate_key': key,
            'region_indices': indices,
            'subset': subset,
            'subset_size': int(len(indices)),
            'member_count_before': int(sum(region['member_count'] for region in subset)),
            'center_dist': float(center_dmat[np.ix_(indices, indices)].max()),
            'vertices': vertices.astype(np.float32),
            'bbox_width_l1': float(widths.sum()),
            'bbox_width_linf': float(widths.max()),
            'proposal_sources': {source_tag},
        }

    for i in range(len(regions)):
        order = np.argsort(center_dmat[i])
        neighbors = [int(j) for j in order[1:1 + int(cfg['proposal_top_center_neighbors'])]]
        for j in neighbors:
            register_subset((i, j), 'center_knn')

        local_pool = neighbors[:int(cfg['proposal_neighbor_pool'])]
        triple_count = 0
        for pos_a, j in enumerate(local_pool):
            for k in local_pool[pos_a + 1:]:
                register_subset((i, j, k), 'local_triple')
                triple_count += 1
                if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                    break
            if triple_count >= int(cfg['proposal_max_triples_per_anchor']):
                break

        quad_count = 0
        for pos_a, j in enumerate(local_pool):
            for pos_b, k in enumerate(local_pool[pos_a + 1:], start=pos_a + 1):
                for l in local_pool[pos_b + 1:]:
                    register_subset((i, j, k, l), 'local_quad')
                    quad_count += 1
                    if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                        break
                if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                    break
            if quad_count >= int(cfg['proposal_max_quads_per_anchor']):
                break

    candidate_df = pd.DataFrame(candidate_map.values()) if candidate_map else pd.DataFrame()
    if candidate_df.empty:
        return candidate_df, candidate_map

    candidate_df['proposal_source_count'] = candidate_df['proposal_sources'].map(len)
    candidate_df['proposal_sources'] = candidate_df['proposal_sources'].map(lambda values: ','.join(sorted(values)))
    candidate_df = candidate_df.sort_values(
        ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'bbox_width_l1'],
        ascending=[False, False, False, True, True],
    ).reset_index(drop=True)
    return candidate_df, candidate_map


def run_iterative_simplex_merging(model, target_label: int, initial_regions, cfg):
    current_regions = [
        {
            **region,
            'center': np.asarray(region['center'], dtype=np.float32),
            'vertices': np.asarray(region['vertices'], dtype=np.float32),
            'member_count': int(region['member_count']),
            'source_ids': list(region['source_ids']),
        }
        for region in initial_regions
    ]
    pass_rows = []
    merge_rows = []

    for pass_idx in range(1, int(cfg['iter_max_passes']) + 1):
        candidate_df, candidate_map = build_local_simplex_candidates(current_regions, cfg)
        if candidate_df.empty:
            pass_rows.append({
                'target_label': int(target_label),
                'pass_idx': pass_idx,
                'regions_start': int(len(current_regions)),
                'candidate_simplexes': 0,
                'candidates_tested': 0,
                'successful_merges': 0,
                'regions_end': int(len(current_regions)),
            })
            break

        shortlisted = pd.concat([
            candidate_df.loc[candidate_df['subset_size'] == 2].head(int(cfg['iter_top_pairs_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 3].head(int(cfg['iter_top_triples_per_pass'])),
            candidate_df.loc[candidate_df['subset_size'] == 4].head(int(cfg['iter_top_quads_per_pass'])),
        ], ignore_index=True)
        shortlisted = shortlisted.sort_values(
            ['subset_size', 'member_count_before', 'proposal_source_count', 'center_dist', 'bbox_width_l1'],
            ascending=[False, False, False, True, True],
        ).reset_index(drop=True)

        certified_candidates = []
        candidates_tested = 0
        for _, row in shortlisted.iterrows():
            candidates_tested += 1
            candidate_obj = candidate_map[row['candidate_key']]
            cert = certify_simplex_region(model, target_label, candidate_obj['vertices'])
            if not cert['certified']:
                continue
            certified_candidates.append({
                **row.to_dict(),
                'cert': cert,
                'candidate_obj': candidate_obj,
            })

        certified_candidates = sorted(
            certified_candidates,
            key=lambda row: (
                -int(row['member_count_before']),
                -int(row['subset_size']),
                -float(row['cert']['min_affine_margin']),
                float(row['center_dist']),
                float(row['bbox_width_l1']),
            ),
        )

        used = set()
        next_regions = []
        successful_merges = 0
        for row in certified_candidates:
            candidate_obj = row['candidate_obj']
            region_indices = tuple(int(idx) for idx in candidate_obj['region_indices'])
            if any(idx in used for idx in region_indices):
                continue
            merged_source_ids = sorted({
                int(source_id)
                for subset_region in candidate_obj['subset']
                for source_id in subset_region['source_ids']
            })
            merged_region = {
                'region_key': f'class{target_label}_pass{pass_idx}_merge{successful_merges + 1}',
                'center': candidate_obj['vertices'].mean(axis=0),
                'vertices': candidate_obj['vertices'].astype(np.float32),
                'member_count': int(candidate_obj['member_count_before']),
                'source_ids': merged_source_ids,
                'cert': row['cert'],
            }
            next_regions.append(merged_region)
            used.update(region_indices)
            successful_merges += 1
            merge_rows.append({
                'target_label': int(target_label),
                'pass_idx': int(pass_idx),
                'candidate_key': row['candidate_key'],
                'subset_size': int(row['subset_size']),
                'member_count_before': int(row['member_count_before']),
                'proposal_source_count': int(row['proposal_source_count']),
                'proposal_sources': row['proposal_sources'],
                'center_dist': float(row['center_dist']),
                'bbox_width_l1': float(row['bbox_width_l1']),
                'bbox_width_linf': float(row['bbox_width_linf']),
                'min_affine_margin': float(row['cert']['min_affine_margin']),
            })

        for idx, region in enumerate(current_regions):
            if idx not in used:
                next_regions.append(region)

        pass_rows.append({
            'target_label': int(target_label),
            'pass_idx': int(pass_idx),
            'regions_start': int(len(current_regions)),
            'candidate_simplexes': int(len(candidate_df)),
            'candidates_tested': int(candidates_tested),
            'successful_merges': int(successful_merges),
            'regions_end': int(len(next_regions)),
        })

        current_regions = next_regions
        if successful_merges == 0:
            break

    return pd.DataFrame(pass_rows), pd.DataFrame(merge_rows), current_regions


CLASS_RESULTS = {}
PASS_TABLES = []
MERGE_TABLES = []
CLASS_SUMMARIES = []

for target_label in sorted(np.unique(Y_SUPPORT)):
    initial_regions = make_initial_center_regions(ATLAS, int(target_label))
    pass_df, merge_df, final_regions = run_iterative_simplex_merging(MODEL, int(target_label), initial_regions, RUN_CFG)
    CLASS_RESULTS[int(target_label)] = {
        'initial_regions': initial_regions,
        'final_regions': final_regions,
        'pass_df': pass_df,
        'merge_df': merge_df,
    }
    PASS_TABLES.append(pass_df)
    if not merge_df.empty:
        MERGE_TABLES.append(merge_df)
    final_member_counts = np.asarray([int(region['member_count']) for region in final_regions], dtype=np.int64)
    CLASS_SUMMARIES.append({
        'target_label': int(target_label),
        'initial_region_count': int(len(initial_regions)),
        'final_region_count': int(len(final_regions)),
        'compression_ratio': float(len(initial_regions) / max(len(final_regions), 1)),
        'successful_merges_total': int(pass_df['successful_merges'].sum()) if not pass_df.empty else 0,
        'passes_executed': int(len(pass_df)),
        'max_member_count_final': int(final_member_counts.max()) if len(final_member_counts) else 0,
        'mean_member_count_final': float(final_member_counts.mean()) if len(final_member_counts) else np.nan,
        'merged_region_fraction_final': float((final_member_counts > 1).mean()) if len(final_member_counts) else np.nan,
    })

PASS_DF = pd.concat(PASS_TABLES, ignore_index=True) if PASS_TABLES else pd.DataFrame()
MERGE_DF = pd.concat(MERGE_TABLES, ignore_index=True) if MERGE_TABLES else pd.DataFrame()
CLASS_SUMMARY_DF = pd.DataFrame(CLASS_SUMMARIES).sort_values('target_label').reset_index(drop=True)

display(CLASS_SUMMARY_DF)
if not PASS_DF.empty:
    display(PASS_DF)
if not MERGE_DF.empty:
    display(MERGE_DF.head(30))


,target_label,initial_region_count,final_region_count,compression_ratio,successful_merges_total,passes_executed,max_member_count_final,mean_member_count_final,merged_region_fraction_final
0,0,600,73,8.219178,360,12,426,8.219178,0.287671
1,1,600,109,5.504587,333,12,388,5.504587,0.174312


,target_label,pass_idx,regions_start,candidate_simplexes,candidates_tested,successful_merges,regions_end
0,0,1,600,18392,1450,97,450
1,0,2,450,13842,1450,15,417
2,0,3,417,12585,1450,24,381
3,0,4,381,11514,1450,15,356
4,0,5,356,10726,1450,37,307
5,0,6,307,9262,1450,17,281
6,0,7,281,8446,1450,52,218
7,0,8,218,6589,1450,21,181
8,0,9,181,5467,1450,12,162
9,0,10,162,4889,1450,39,117


,target_label,pass_idx,candidate_key,subset_size,member_count_before,proposal_source_count,proposal_sources,center_dist,bbox_width_l1,bbox_width_linf,min_affine_margin
0,0,1,class0_poly_208|class0_poly_355|class0_poly_39...,4,4,1,local_quad,1.510312,1.760006,1.434673,6.934531
1,0,1,class0_poly_56|class0_poly_142|class0_poly_397...,4,4,1,local_quad,2.941667,3.680311,2.144470,6.190917
2,0,1,class0_poly_127|class0_poly_146|class0_poly_16...,4,4,1,local_quad,3.403719,3.630636,1.664628,6.106846
3,0,1,class0_poly_129|class0_poly_191|class0_poly_43...,4,4,1,local_quad,2.828361,3.115967,1.000000,5.980681
4,0,1,class0_poly_75|class0_poly_200|class0_poly_240...,4,4,1,local_quad,2.907691,4.394903,2.503358,5.843037
5,0,1,class0_poly_64|class0_poly_201|class0_poly_264...,4,4,1,local_quad,2.717875,3.436587,1.000000,4.173045
6,0,1,class0_poly_71|class0_poly_439|class0_poly_515...,4,4,1,local_quad,3.147385,3.344172,1.000000,3.812233
7,0,1,class0_poly_31|class0_poly_55|class0_poly_136|...,4,4,1,local_quad,3.143034,3.682184,1.000000,3.656785
8,0,1,class0_poly_193|class0_poly_196|class0_poly_29...,4,4,1,local_quad,3.414135,5.489774,1.111579,2.393121
9,0,1,class0_poly_35|class0_poly_171|class0_poly_419...,4,4,1,local_quad,3.409623,4.778126,1.000000,2.359557


In [6]:
GLOBAL_SUMMARY_DF = pd.DataFrame([{
    'atlas_build_seconds': float(ATLAS_BUILD_SECONDS),
    'total_initial_regions': int(CLASS_SUMMARY_DF['initial_region_count'].sum()),
    'total_final_regions': int(CLASS_SUMMARY_DF['final_region_count'].sum()),
    'global_compression_ratio': float(CLASS_SUMMARY_DF['initial_region_count'].sum() / max(CLASS_SUMMARY_DF['final_region_count'].sum(), 1)),
    'total_successful_merges': int(CLASS_SUMMARY_DF['successful_merges_total'].sum()),
    'max_member_count_overall': int(CLASS_SUMMARY_DF['max_member_count_final'].max()),
    'mean_member_count_overall': float(CLASS_SUMMARY_DF['mean_member_count_final'].mean()),
    'merged_region_fraction_overall': float(CLASS_SUMMARY_DF['merged_region_fraction_final'].mean()),
}])

display(GLOBAL_SUMMARY_DF)
print('\nPer-class summary:')
display(CLASS_SUMMARY_DF)

if not MERGE_DF.empty:
    print('\nMerge-level summary:')
    display(
        MERGE_DF.groupby(['target_label', 'subset_size'])[['member_count_before', 'center_dist', 'bbox_width_l1', 'min_affine_margin']]
        .agg(['count', 'mean', 'median'])
    )
    print('\nLargest certified merged simplexes:')
    display(
        MERGE_DF.sort_values(['member_count_before', 'subset_size', 'min_affine_margin'], ascending=[False, False, False]).head(20)
    )
else:
    print('No certified simplex merges were found under the current Adult-space settings.')


,atlas_build_seconds,total_initial_regions,total_final_regions,global_compression_ratio,total_successful_merges,max_member_count_overall,mean_member_count_overall,merged_region_fraction_overall
0,14.761443,1200,182,6.593407,693,426,6.861883,0.230992



Per-class summary:


,target_label,initial_region_count,final_region_count,compression_ratio,successful_merges_total,passes_executed,max_member_count_final,mean_member_count_final,merged_region_fraction_final
0,0,600,73,8.219178,360,12,426,8.219178,0.287671
1,1,600,109,5.504587,333,12,388,5.504587,0.174312



Merge-level summary:


member_count_before                   center_dist  \
                                       count       mean median       count   
target_label subset_size                                                     
0            2                           247   5.493927    2.0         247   
             3                            59  17.101695    6.0          59   
             4                            54  37.518519   11.5          54   
1            2                           218   8.004587    2.0         218   
             3                            72  24.097222    6.0          72   
             4                            43  18.534884    9.0          43   

                                             bbox_width_l1            \
                              mean    median         count      mean   
target_label subset_size                                               
0            2            5.843328  5.466173           247  5.843328   
             3            7.502528  7.971718            59  9.423423   
             4            6.723597  6.311384            54  9.678815   
1            2            4.537293  4.357768           218  4.537293   
             3            5.665483  5.037406            72  6.964630   
             4            5.003865  2.634062            43  6.738052   

                                    min_affine_margin                      
                             median             count      mean    median  
target_label subset_size                                                   
0            2             5.466173               247  2.366848  1.719133  
             3            10.206985                59  1.929792  0.979040  
             4             8.720174                54  2.670981  2.153973  
1            2             4.357768               218  1.183617  0.620040  
             3             6.501719                72  1.007836  0.445569  
             4             3.800377                43  1.205722  0.752601


Largest certified merged simplexes:


,target_label,pass_idx,candidate_key,subset_size,member_count_before,proposal_source_count,proposal_sources,center_dist,bbox_width_l1,bbox_width_linf,min_affine_margin
346,0,12,class0_pass11_merge1|class0_pass11_merge4|clas...,4,426,1,local_quad,11.343153,18.236965,1.441660,2.045391
329,0,11,class0_pass10_merge1|class0_pass10_merge2|clas...,4,403,1,local_quad,12.364035,18.820465,1.649004,0.042823
682,1,12,class1_pass11_merge1|class1_pass11_merge3,2,388,1,center_knn,11.001366,11.001366,4.397873,0.829754
670,1,11,class1_pass10_merge1|class1_pass10_merge3|clas...,3,379,1,local_triple,10.956229,12.333587,1.240424,0.144029
290,0,10,class0_pass9_merge1|class0_pass9_merge2|class0...,4,375,1,local_quad,10.365749,17.359808,1.559200,0.044775
647,1,10,class1_pass9_merge1|class1_pass9_merge3|class1...,3,335,1,local_triple,6.318826,9.328991,2.201160,0.448093
278,0,9,class0_pass8_merge1|class0_pass8_merge2|class0...,3,321,1,local_triple,11.350571,13.223268,1.443686,0.017951
632,1,9,class1_pass8_merge1|class1_pass8_merge2,2,293,1,center_knn,9.095755,9.095755,4.402320,1.043114
612,1,8,class1_pass7_merge1|class1_pass7_merge3|class1...,3,251,1,local_triple,6.193054,8.287340,0.998250,0.356618
257,0,8,class0_pass7_merge1|class0_pass7_merge3|class0...,3,244,1,local_triple,13.353945,15.054073,1.338352,0.152141


## Questions to ask after running

- Does the simplex representation certify merges on Adult where axis-aligned boxes failed completely?
- Is compression now nonzero on one or both target classes?
- Do pairs/triples/quads behave differently in certification success?
- If simplex merging still fails broadly, is the issue the fresh-certification conservatism rather than the wrapper geometry itself?
